# Commands — Hooks & Bisect
> *Hooks = code police that runs automatically.*
> *Bisect = detective work to find who broke production.* 🔍

---

> 💬 Scenario A: Someone on your team pushes code that fails style checks.
> Every code review has the same comment: "remove trailing whitespace / fix formatting."
> Hooks automate the checking so humans don't have to.
>
> Scenario B: Tests were passing 2 weeks ago. Now they're failing.
> 50 commits happened in between. Nobody confesses.
> Bisect figures out which commit killed it — in 6 checks, not 50.

## 🪝 Git Hooks — Automated Standards Enforcement

Hooks are **shell scripts that run automatically** at specific Git events.
They live in `.git/hooks/` — every repo already has sample files there.

| Hook | When it runs | What people use it for |
|------|-------------|----------------------|
| `pre-commit` | Before `git commit` | Run linter, formatter, auto-check for debug statements |
| `commit-msg` | After you type commit message | Enforce message format (must start with `feat:` or `fix:`) |
| `pre-push` | Before `git push` | Run test suite — block push if tests fail |
| `post-merge` | After `git pull/merge` | Auto-run `npm install` so you don't wonder why packages are missing |
| `prepare-commit-msg` | Before editor opens | Auto-insert branch name or ticket number |

> 💬 Without hooks: Developer pushes broken code. CI fails. 15 minutes wasted.
> With hooks: Code is checked at commit time. Problem found in 2 seconds. Never pushed.
> The meeting where someone says "why did this broken code get pushed?" never happens.

## 🛠️ Creating a Hook (Manual Method)

```bash
# See what sample hooks already exist:
ls .git/hooks/
# pre-commit.sample  commit-msg.sample  pre-push.sample  ...

# Create a pre-commit hook that runs linting before every commit:
# (Git Bash / Linux / Mac):
cat > .git/hooks/pre-commit << 'EOF'
#!/bin/sh
echo "Running linter before commit..."
flake8 .
if [ $? -ne 0 ]; then
  echo "Linting failed. Please fix errors before committing."
  exit 1
fi
echo "All good. Committing."
EOF

chmod +x .git/hooks/pre-commit   # Mac/Linux: make it executable

# Test it — try to commit with messy code:
git commit -m "test"
# If linting fails → commit is BLOCKED. Fix the code first.
```

> ⚠️ `.git/hooks/` is **NOT committed to the repo** — these only exist on your machine.
> To share hooks with your whole team, install the `pre-commit` framework (next section).

## 📦 The `pre-commit` Framework — Team-Grade Hook Management

This is the professional way. One config file, everyone gets the same hooks automatically.

```bash
# Install the pre-commit tool:
pip install pre-commit
```

Create `.pre-commit-config.yaml` in your repo root:
```yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace      # No trailing spaces
      - id: end-of-file-fixer        # Every file ends with newline
      - id: check-yaml               # No broken YAML files
      - id: check-merge-conflict     # No unresolved conflict markers committed
  - repo: https://github.com/psf/black
    rev: 23.12.0
    hooks:
      - id: black                    # Python auto-formatter
```

```bash
# Activate for your local checkout:
pre-commit install
# Now runs automatically on every commit!

# Run manually on all files (useful for first setup):
pre-commit run --all-files
```

> 💬 The `.pre-commit-config.yaml` file IS committed to the repo.
> Everyone who clones the repo and runs `pre-commit install` gets the same hooks.
> The whole team gets the code police. Nobody gets special treatment.

## 🕵️ `git bisect` — Binary Search for the Breaking Commit

**Scenario:** Your test suite passed last Thursday. It fails today. 50 commits happened.
Which commit broke it? You're not reading 50 full diffs by hand.

`git bisect` uses **binary search** — it checks the MIDDLE commit, you tell it good or bad,
it eliminates half the rest. 50 commits = approximately 6 checks to find the culprit.

```bash
# 1. Start the investigation:
git bisect start

# 2. Tell it: right now is broken:
git bisect bad

# 3. Tell it: this old commit was fine (use a tag, hash, or relative ref):
git bisect good v1.2.0
git bisect good abc1234
# Git checks out the middle commit automatically. You're now in history.

# 4. Test — does the bug exist here?
#    YES → still broken:
git bisect bad
#    NO → this version was fine:
git bisect good
# Git keeps narrowing down. Repeat until:
# "b3a7f12 is the first bad commit"

# 5. Inspect the finding:
git show b3a7f12
# Read the diff. Find the bug. Know who committed it.

# 6. Return to HEAD:
git bisect reset
```

> 💬 The best part of git bisect:
> "Who introduced this bug?" now has a specific commit hash and author.
> Whether you use that information for fixing silently or for a pointed PR comment
> is entirely up to you and your relationship with that teammate.

## 🤖 Automated Bisect (Let Git Do the Testing)

If you have a test command that returns 0 for pass and non-zero for fail, Git can bisect completely automatically.

```bash
git bisect start
git bisect bad HEAD
git bisect good v1.0.0

# Provide a test script — Git runs it on each commit:
git bisect run python -m pytest tests/test_login.py
# OR:
git bisect run sh ./scripts/test.sh

# Git runs through every candidate commit, checks pass/fail, and tells you:
# "abc1234 is the first bad commit" — with zero manual checking needed

git bisect reset   # All done
```

> 💬 Automated bisect is what separates "I'll look into this" from "I found the commit in 2 minutes."
> Set up a focused test that triggers the bug. Run bisect. Get coffee.
> Come back to a solved mystery.